In [ ]:
# Cell 1: Setup and Imports
import os
import json
import re
import numpy as np
import requests
from bs4 import BeautifulSoup
from markdownify import markdownify as md
from sentence_transformers import SentenceTransformer
from typing import List, Dict, Any
import chromadb

# Set your API key here
os.environ["OPENROUTER_API_KEY"] = "REDACTED_OPENROUTER_API_KEY"

print("✅ All imports loaded successfully")

✅ All imports loaded successfully


In [13]:
# Cell 2: Load chunks and initialize RAG system

# Load chunks from your existing file
with open("embeddings.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"✅ Loaded {len(chunks)} chunks")

# Initialize the embedding model
model = SentenceTransformer("intfloat/multilingual-e5-base")
print("✅ Model loaded")

✅ Loaded 16 chunks


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4508.35it/s]


✅ Model loaded


In [16]:
# Cell 3: RAG System with Citations (Improved - Forces Using Available Context)
import re

class RAGWithCitations:
    def __init__(self, chunks: List[Dict], model):
        self.chunks = chunks
        self.model = model
        self.api_key = os.getenv("OPENROUTER_API_KEY")
    
    def retrieve_top_k(self, query: str, k: int = 7) -> List[Dict]:
        """Retrieve top-k relevant chunks with citation info"""
        question_embedding = self.model.encode(
            "query: " + query,
            normalize_embeddings=True
        )
        
        results = []
        for chunk in self.chunks:
            chunk_embedding = np.array(chunk["embedding"])
            similarity = np.dot(question_embedding, chunk_embedding)
            results.append({
                "chunk": chunk,
                "score": float(similarity),
                "source": chunk.get("title", "Unknown")
            })
        
        results.sort(key=lambda x: x["score"], reverse=True)
        return results[:k]
    
    def prepare_context_with_citations(self, retrieved: List[Dict]) -> tuple:
        """Format retrieved chunks with citation markers"""
        context_with_citations = []
        citations = []
        
        for i, result in enumerate(retrieved, 1):
            chunk_text = result["chunk"]["text"]
            source = result["source"]
            score = result["score"]
            
            # Add citation marker with clear separation
            context_with_citations.append(f"[{i}] {chunk_text}")
            
            citations.append({
                "id": i,
                "source": source,
                "text": chunk_text[:300] + "..." if len(chunk_text) > 300 else chunk_text,
                "relevance_score": round(score, 4)
            })
        
        return context_with_citations, citations
    
    def detect_language(self, text: str) -> str:
        """Detect if text is Arabic or English"""
        if any('\u0600' <= c <= '\u06FF' for c in text):
            return "arabic"
        return "english"
    
    def generate_response(self, query: str, context_with_citations: List[str], citations: List[Dict]) -> Dict[str, Any]:
        """Generate response with citations using LLM"""
        
        context_str = "\n\n".join(context_with_citations)
        lang = self.detect_language(query)
        
        # Debug: Print what we're sending
        print(f"\n🔍 Context being sent to LLM:\n{context_str}\n")
        
        if lang == "arabic":
            prompt = f"""أنت مساعد Telecom Egypt الذكي. أجب عن سؤال المستخدم بناءً على السياق المقدم فقط.

السياق (مع الاستشهادات):
{context_str}

سؤال المستخدم: {query}

تعليمات مهمة جداً:
1. أجب فقط بناءً على السياق المقدم. إذا كان هناك معلومات متوفرة في السياق، استخدمها.
2. استخدم الاستشهادات مثل [1]، [2] للإشارة إلى المصادر التي استخدمتها
3. إذا لم يحتوي السياق على الإجابة، قل "لا يمكنني العثور على هذه المعلومات في المستندات المتوفرة"
4. كن مفيداً ومحترفاً
5. اذكر جميع المصادر التي استخدمتها في الإجابة

الإجابة:"""
        else:
            prompt = f"""You are Telecom Egypt's intelligent assistant. Answer the user's question based ONLY on the provided context.

Context (with citations):
{context_str}

User Question: {query}

Very Important Instructions:
1. Answer based ONLY on the context provided. If there is information available in the context, USE IT.
2. Use citations like [1], [2] to reference sources you used
3. If the context doesn't contain the answer, say "I cannot find this information in the provided documents"
4. Be helpful and professional
5. List all the sources you used in your answer

Answer:"""
        
        print(f"\n📝 Sending prompt to LLM...")
        
        response = requests.post(
            "https://openrouter.ai/api/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {self.api_key}",
                "Content-Type": "application/json"
            },
            json={
                "model": "openai/gpt-4o-mini",
                "messages": [{"role": "user", "content": prompt}]
            }
        )
        
        if response.status_code == 200:
            data = response.json()
            answer = data["choices"][0]["message"]["content"]
        else:
            answer = f"Error: {response.status_code} - {response.text}"
        
        # Extract which citations were used in the response
        citation_pattern = r'\[(\d+)\]'
        used_citation_ids = set(map(int, re.findall(citation_pattern, answer)))
        used_citations = [c for c in citations if c['id'] in used_citation_ids]
        
        # If no citations were used but the response doesn't say "cannot find", 
        # check if the answer references any source names
        if not used_citations and "لا يمكنني" not in answer and "cannot find" not in answer:
            # Try to find source mentions without citation markers
            for citation in citations:
                if citation['source'].lower() in answer.lower():
                    used_citations.append(citation)
                    break
        
        return {
            "query": query,
            "response": answer,
            "citations": used_citations,
            "all_citations": citations,
            "language_detected": lang
        }
    
    def query(self, question: str, k: int = 7) -> Dict[str, Any]:
        """Complete RAG pipeline with citations"""
        retrieved = self.retrieve_top_k(question, k=k)
        context_with_citations, citations = self.prepare_context_with_citations(retrieved)
        result = self.generate_response(question, context_with_citations, citations)
        result["retrieved_chunks"] = retrieved
        return result

print("✅ RAG class defined (improved with more chunks and better prompting)")

✅ RAG class defined (improved with more chunks and better prompting)


In [ ]:
# Cell 4: Test with sample queries

rag = RAGWithCitations(chunks, model)

test_queries = [
    "كيف يمكنني دفع الفاتورة؟",
    "What is Bill Limit service?",
    "ازاي أعرف رصيدي؟"
]

for query in test_queries:
    print("\n" + "="*60)
    print(f"Query: {query}")
    print("="*60)
    
    result = rag.query(query, k=5)  # Increase to 5 to get more context
    
    print("\nTop Retrieved Chunks:")
    for i, chunk in enumerate(result["retrieved_chunks"][:5], 1):
        print(f"  [{i}] Score: {chunk['score']:.4f} - Source: {chunk['source']}")
    
    print(f"\nResponse:\n{result['response']}\n")
    
    if result["citations"]:
        print("Citations Used:")
        for citation in result["citations"]:
            print(f"  [{citation['id']}] {citation['source']} (Score: {citation['relevance_score']:.4f})")
            print(f"     Excerpt: {citation['text'][:150]}...")
    else:
        print("No citations found in the response.")


Query: كيف يمكنني دفع الفاتورة؟

🔍 Context being sent to LLM:
[1] يمكنك اختيار حد مسبق لفاتورتك بقيمة معينة لإشتراكك الشهري باستخدام Bill Limit. للإشتراك اطلب *883#

[2] يمكنك الإستعلام عن رصيدك من خلال الإتصال بـ #550* وسوف تصلك رسالة مؤقتة تظهر لك المبلغ المتبقي من رصيدك.

[3] يمكنك معرفة الأرقام الغير متاحة أو مغلقة واصبحت متاحة عن طريق رسالة نصية لإخطارك بمعاودة الإتصال. للإشتراك اطلب *066#

[4] من خلال خدمة البريد الصوتي سيتم استقبال مكالمتك في حالة عدم الوصول إليك. للإشتراك اطلب *505#

[5] ## 3. Fawry Services
- Visit any Fawry outlet
- Provide your WE account number
- Pay the amount in cash

## 4. Bank Transfer
- Transfer the amount to WE's bank account
- Include your account number as reference
- Allow 1-2 business days for processing

## 5. ATM Payment
- Use any ATM with bill payment feature
- Select Telecom Egypt as the service provider
- Enter your account number and payment amount


📝 Sending prompt to LLM...

Top Retrieved Chunks:
  [1] Score: 0.8147 - Source: اتحكم فى فا

: 